In [1]:
!pip install xarray
!pip install netCDF4
!pip install dask

In [2]:
from pathlib import Path
import xarray as xr
import pandas as pd
base_dir = Path(".")  

all_nc = list(base_dir.rglob("*.nc"))

rows = []

for file in all_nc:
    ds = xr.open_dataset(file, engine='netcdf4')

    time_name = "valid_time" if "valid_time" in ds.coords else "time"
    times = pd.to_datetime(ds[time_name].values)

    rows.append({
        "file": str(file),
        "type": "accum" if "accum" in file.name else "instant" if "instant" in file.name else "other",
        "start": times.min(),
        "end": times.max(),
        "years": sorted(set(times.year)),
        "months": sorted(set(times.month)),
        "n_times": len(times)
    })

files_info = pd.DataFrame(rows)
files_info

,file,type,start,end,years,months,n_times
0,dane_pogoda/4eb9508671f184b580910da9b60a2189/d...,instant,2021-05-01,2021-09-30 23:00:00,[2021],"[5, 6, 7, 8, 9]",3672
1,dane_pogoda/4eb9508671f184b580910da9b60a2189/d...,accum,2021-05-01,2021-09-30 23:00:00,[2021],"[5, 6, 7, 8, 9]",3672
2,dane_pogoda/391f004915db8ebcb01569bfd11b7247/d...,instant,2020-05-01,2020-09-30 23:00:00,[2020],"[5, 6, 7, 8, 9]",3672
3,dane_pogoda/391f004915db8ebcb01569bfd11b7247/d...,accum,2020-05-01,2020-09-30 23:00:00,[2020],"[5, 6, 7, 8, 9]",3672
4,dane_pogoda/a410b01a0b3d4ee1b01b661ca754a713/d...,instant,2019-05-01,2019-09-30 23:00:00,[2019],"[5, 6, 7, 8, 9]",3672
5,dane_pogoda/a410b01a0b3d4ee1b01b661ca754a713/d...,accum,2019-05-01,2019-09-30 23:00:00,[2019],"[5, 6, 7, 8, 9]",3672
6,dane_pogoda/1a74942c3f0dc3b851ebb49f704c1ab4/d...,instant,2024-05-01,2024-09-30 23:00:00,[2024],"[5, 6, 7, 8, 9]",3672
7,dane_pogoda/1a74942c3f0dc3b851ebb49f704c1ab4/d...,accum,2024-05-01,2024-09-30 23:00:00,[2024],"[5, 6, 7, 8, 9]",3672
8,dane_pogoda/831893f88cc0000e23748e601b66b60a/d...,instant,2023-05-01,2023-09-30 23:00:00,[2023],"[5, 6, 7, 8, 9]",3672
9,dane_pogoda/831893f88cc0000e23748e601b66b60a/d...,accum,2023-05-01,2023-09-30 23:00:00,[2023],"[5, 6, 7, 8, 9]",3672


In [3]:
files_info.sort_values(["type", "start", "end"])

,file,type,start,end,years,months,n_times
5,dane_pogoda/a410b01a0b3d4ee1b01b661ca754a713/d...,accum,2019-05-01,2019-09-30 23:00:00,[2019],"[5, 6, 7, 8, 9]",3672
3,dane_pogoda/391f004915db8ebcb01569bfd11b7247/d...,accum,2020-05-01,2020-09-30 23:00:00,[2020],"[5, 6, 7, 8, 9]",3672
1,dane_pogoda/4eb9508671f184b580910da9b60a2189/d...,accum,2021-05-01,2021-09-30 23:00:00,[2021],"[5, 6, 7, 8, 9]",3672
11,dane_pogoda/714a357dc7a31f2b65486abcd83b3c47/d...,accum,2022-05-01,2022-09-30 23:00:00,[2022],"[5, 6, 7, 8, 9]",3672
9,dane_pogoda/831893f88cc0000e23748e601b66b60a/d...,accum,2023-05-01,2023-09-30 23:00:00,[2023],"[5, 6, 7, 8, 9]",3672
7,dane_pogoda/1a74942c3f0dc3b851ebb49f704c1ab4/d...,accum,2024-05-01,2024-09-30 23:00:00,[2024],"[5, 6, 7, 8, 9]",3672
13,dane_pogoda/3dd81707afb59795bc103f9cb7b7cb61/d...,accum,2025-05-01,2025-09-30 23:00:00,[2025],"[5, 6, 7, 8, 9]",3672
4,dane_pogoda/a410b01a0b3d4ee1b01b661ca754a713/d...,instant,2019-05-01,2019-09-30 23:00:00,[2019],"[5, 6, 7, 8, 9]",3672
2,dane_pogoda/391f004915db8ebcb01569bfd11b7247/d...,instant,2020-05-01,2020-09-30 23:00:00,[2020],"[5, 6, 7, 8, 9]",3672
0,dane_pogoda/4eb9508671f184b580910da9b60a2189/d...,instant,2021-05-01,2021-09-30 23:00:00,[2021],"[5, 6, 7, 8, 9]",3672


In [4]:
selected_files = files_info[
    files_info["years"].apply(lambda yrs: any(2019 <= y <= 2025 for y in yrs))
]["file"].tolist()

instant_files = [f for f in selected_files if "instant" in f]
accum_files = [f for f in selected_files if "accum" in f]

print("selected:", len(selected_files))
print("instant:", len(instant_files))
print("accum:", len(accum_files))

if instant_files and accum_files:
    ds_instant = xr.open_mfdataset(instant_files, engine='netcdf4', combine='by_coords')
    ds_accum = xr.open_mfdataset(accum_files, engine='netcdf4', combine='by_coords')

selected: 14
instant: 7
accum: 7


In [5]:
print(ds_instant)
print(ds_accum)

<xarray.Dataset> Size: 1MB
Dimensions:     (valid_time: 25704, latitude: 1, longitude: 2)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 206kB 2019-05-01 ... 2025-09-30T2...
    expver      (valid_time) <U4 411kB dask.array<chunksize=(3672,), meta=np.ndarray>
  * latitude    (latitude) float64 8B 49.5
  * longitude   (longitude) float64 16B 20.75 21.0
    number      int64 8B 0
Data variables:
    u10         (valid_time, latitude, longitude) float32 206kB dask.array<chunksize=(3672, 1, 2), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 206kB dask.array<chunksize=(3672, 1, 2), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 206kB dask.array<chunksize=(3672, 1, 2), meta=np.ndarray>
    sp          (valid_time, latitude, longitude) float32 206kB dask.array<chunksize=(3672, 1, 2), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
   

In [6]:
lat = 49.39
lon = 20.87

inst = ds_instant.sel(latitude=lat, longitude=lon, method="nearest")
acc = ds_accum.sel(latitude=lat, longitude=lon, method="nearest")

In [7]:
df_inst = inst.to_dataframe().reset_index()
df_acc = acc.to_dataframe().reset_index()

In [8]:
print(df_inst.columns)
print(df_acc.columns)

Index(['valid_time', 'u10', 'v10', 't2m', 'sp', 'number', 'latitude',
       'longitude', 'expver'],
      dtype='str')
Index(['valid_time', 'tp', 'number', 'latitude', 'longitude', 'expver'], dtype='str')


In [9]:
df_inst

,valid_time,u10,v10,t2m,sp,number,latitude,longitude,expver
0,2019-05-01 00:00:00,1.419220,-2.990616,281.554321,94163.281250,0,49.5,20.75,0001
1,2019-05-01 01:00:00,1.756149,-3.021606,281.316040,94133.046875,0,49.5,20.75,0001
2,2019-05-01 02:00:00,1.935593,-3.079102,281.119507,94126.218750,0,49.5,20.75,0001
3,2019-05-01 03:00:00,2.064880,-3.207886,280.828308,94087.937500,0,49.5,20.75,0001
4,2019-05-01 04:00:00,2.056519,-3.051651,280.693054,94096.718750,0,49.5,20.75,0001
...,...,...,...,...,...,...,...,...,...
25699,2025-09-30 19:00:00,-0.133713,-1.783752,279.393555,95745.078125,0,49.5,20.75,0001
25700,2025-09-30 20:00:00,-0.528625,-2.117447,279.219299,95751.812500,0,49.5,20.75,0001
25701,2025-09-30 21:00:00,-0.633621,-2.245499,279.408295,95740.562500,0,49.5,20.75,0001
25702,2025-09-30 22:00:00,-0.486664,-2.207901,279.213501,95722.203125,0,49.5,20.75,0001


In [10]:
df = df_inst.merge(df_acc, on="valid_time")

In [11]:
import numpy as np

df["temp"] = df["t2m"] - 273.15
df["wspd"] = np.sqrt(df["u10"]**2 + df["v10"]**2)
df["prcp"] = df["tp"] * 1000
df["pres"] = df["sp"] / 100  # Pa -> hPa

In [12]:
df

,valid_time,u10,v10,t2m,sp,number_x,latitude_x,longitude_x,expver_x,tp,number_y,latitude_y,longitude_y,expver_y,temp,wspd,prcp,pres
0,2019-05-01 00:00:00,1.419220,-2.990616,281.554321,94163.281250,0,49.5,20.75,0001,0.000179,0,49.5,20.75,0001,8.404327,3.310282,0.178814,941.632812
1,2019-05-01 01:00:00,1.756149,-3.021606,281.316040,94133.046875,0,49.5,20.75,0001,0.000156,0,49.5,20.75,0001,8.166046,3.494877,0.156403,941.330444
2,2019-05-01 02:00:00,1.935593,-3.079102,281.119507,94126.218750,0,49.5,20.75,0001,0.000191,0,49.5,20.75,0001,7.969513,3.636947,0.191212,941.262207
3,2019-05-01 03:00:00,2.064880,-3.207886,280.828308,94087.937500,0,49.5,20.75,0001,0.000128,0,49.5,20.75,0001,7.678314,3.815005,0.128269,940.879395
4,2019-05-01 04:00:00,2.056519,-3.051651,280.693054,94096.718750,0,49.5,20.75,0001,0.000025,0,49.5,20.75,0001,7.543060,3.679924,0.025272,940.967163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25699,2025-09-30 19:00:00,-0.133713,-1.783752,279.393555,95745.078125,0,49.5,20.75,0001,0.000170,0,49.5,20.75,0001,6.243561,1.788757,0.170231,957.450806
25700,2025-09-30 20:00:00,-0.528625,-2.117447,279.219299,95751.812500,0,49.5,20.75,0001,0.000138,0,49.5,20.75,0001,6.069305,2.182436,0.137806,957.518127
25701,2025-09-30 21:00:00,-0.633621,-2.245499,279.408295,95740.562500,0,49.5,20.75,0001,0.000204,0,49.5,20.75,0001,6.258301,2.333182,0.203609,957.405640
25702,2025-09-30 22:00:00,-0.486664,-2.207901,279.213501,95722.203125,0,49.5,20.75,0001,0.000209,0,49.5,20.75,0001,6.063507,2.260900,0.208855,957.222046


In [13]:
time_col = "time" if "time" in df.columns else "valid_time"

df[time_col] = pd.to_datetime(df[time_col])
df["year"] = df[time_col].dt.isocalendar().year
df["week"] = df[time_col].dt.isocalendar().week

df_weekly = df.groupby(["year", "week"], as_index=False)[["temp", "wspd", "prcp", "pres"]].mean()

df_weekly_filtered = df_weekly[(df_weekly["year"] >= 2019) & (df_weekly["year"] <= 2025)]
df_weekly_filtered.head()

,year,week,temp,wspd,prcp,pres
0,2019,18,8.644572,2.952617,0.247276,940.725464
1,2019,19,8.801729,2.939627,0.163314,946.074036
2,2019,20,10.284056,2.588411,0.200703,949.088623
3,2019,21,13.292661,2.689059,0.371334,943.737183
4,2019,22,14.097325,2.743598,0.164248,950.881531


In [14]:
df_weekly_filtered.tail()

,year,week,temp,wspd,prcp,pres
155,2025,36,18.315823,1.842087,0.052185,951.153442
156,2025,37,16.005041,2.150433,0.217361,950.720215
157,2025,38,15.537229,2.754735,0.031596,954.758118
158,2025,39,10.943379,2.692644,0.154106,955.596375
159,2025,40,6.669147,2.374559,0.304888,955.025818


In [15]:
df_weekly_filtered.to_csv("df_weekly_2019_2025.csv", index=False)

In [16]:
df_hourly = df.copy()[["valid_time", "temp", "wspd", "prcp", "pres"]]
df_hourly["week"] = df_hourly[time_col].dt.isocalendar().week

df_hourly_filtered = df_hourly[
    (df_hourly[time_col].dt.year >= 2019) &
    (df_hourly[time_col].dt.year <= 2025)
]

df_hourly_filtered.head()

,valid_time,temp,wspd,prcp,pres,week
0,2019-05-01 00:00:00,8.404327,3.310282,0.178814,941.632812,18
1,2019-05-01 01:00:00,8.166046,3.494877,0.156403,941.330444,18
2,2019-05-01 02:00:00,7.969513,3.636947,0.191212,941.262207,18
3,2019-05-01 03:00:00,7.678314,3.815005,0.128269,940.879395,18
4,2019-05-01 04:00:00,7.543060,3.679924,0.025272,940.967163,18


In [17]:
df_hourly_filtered.to_csv("df_hourly_2019_2025.csv", index=False)